# 01 — kitpri_v2 Dataset Exploration

Explore the kitpri_v2 dataset: class balance, metadata, waveform samples, and
log-mel spectrogram visualizations.

**Dataset stats (from Kaggle):**
- Cooking:    1,784 clips  (`aug_c_XXXXX.wav`)
- Noncooking: 1,724 clips  (`aug_n_XXXXX.wav`)
- Total:      3,508 clips
- Train / Val / Test ≈ 70 / 15 / 15 (stratified)

**Audio format:** mono WAV, 32 kHz, 10 s clips

**Splits (after filtering for existing files on Kaggle):**
| Split | Total | Cooking | Noncooking |
|---|---:|---:|---:|
| Train | 2,428 | 1,235 | 1,193 |
| Val   |   531 |   276 |   255 |
| Test  |   549 |   273 |   276 |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Adjust this to point at your local data root ─────────────────────────────
DATA_ROOT = Path('../data/raw/kitpri-v2')
AUDIO_DIR = DATA_ROOT / 'audio_32k'

train_df = pd.read_csv(DATA_ROOT / 'metadata/train.csv')
val_df   = pd.read_csv(DATA_ROOT / 'metadata/val.csv')
test_df  = pd.read_csv(DATA_ROOT / 'metadata/test.csv')

print(f'Train : {len(train_df):>5}  | cooking={( train_df["label"]==1).sum():>4}  noncooking={(train_df["label"]==0).sum():>4}')
print(f'Val   : {len(val_df):>5}  | cooking={( val_df["label"]==1).sum():>4}  noncooking={(val_df["label"]==0).sum():>4}')
print(f'Test  : {len(test_df):>5}  | cooking={( test_df["label"]==1).sum():>4}  noncooking={(test_df["label"]==0).sum():>4}')
train_df.head()

In [ ]:
# Class balance bar chart
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, (df, title) in zip(axes, [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]):
    counts = df['label'].map({0: 'noncooking', 1: 'cooking'}).value_counts()
    ax.bar(counts.index, counts.values, color=['#2196F3', '#FF5722'], width=0.5)
    ax.set_title(f'{title} split  (n={len(df)})')
    ax.set_ylabel('Clip count')
    for i, (k, v) in enumerate(counts.items()):
        ax.text(i, v + 5, str(v), ha='center', fontsize=11)
plt.suptitle('kitpri_v2 — Class Distribution per Split', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Waveform and log-mel spectrogram comparison
import librosa
import librosa.display

SR        = 32_000
DURATION  = 10
N_MELS    = 64
HOP       = 320
N_FFT     = 1024
F_MIN     = 50
F_MAX     = 14_000

sample_cooking    = train_df[train_df['label'] == 1].iloc[0]
sample_noncooking = train_df[train_df['label'] == 0].iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

for row_idx, (sample, label_name) in enumerate([
    (sample_cooking, 'Cooking'), (sample_noncooking, 'Non-cooking')
]):
    fpath = DATA_ROOT / sample['file_path']
    wav, _ = librosa.load(str(fpath), sr=SR, mono=True, duration=DURATION)

    # Waveform
    ax_wav = axes[row_idx, 0]
    ax_wav.plot(np.linspace(0, DURATION, len(wav)), wav, linewidth=0.4)
    ax_wav.set_title(f'{label_name} — waveform')
    ax_wav.set_xlabel('Time (s)')
    ax_wav.set_ylabel('Amplitude')

    # Log-mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
        fmin=F_MIN, fmax=F_MAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    ax_mel = axes[row_idx, 1]
    img = librosa.display.specshow(
        mel_db, sr=SR, hop_length=HOP, fmin=F_MIN, fmax=F_MAX,
        x_axis='time', y_axis='mel', ax=ax_mel, cmap='magma'
    )
    ax_mel.set_title(f'{label_name} — log-mel (64 bins)')
    fig.colorbar(img, ax=ax_mel, format='%+2.0f dB')

plt.tight_layout()
plt.show()

In [ ]:
# Source filename prefix analysis
# Cooking clips: aug_c_XXXXX.wav  | Noncooking: aug_n_XXXXX.wav
train_df['prefix'] = train_df['file_path'].apply(
    lambda p: Path(p).name[:5]
)
print('Unique prefixes:', train_df['prefix'].unique())
print('\nSample cooking clips:')
print(train_df[train_df['label'] == 1]['file_path'].head(5).values)
print('\nSample non-cooking clips:')
print(train_df[train_df['label'] == 0]['file_path'].head(5).values)